<a href="https://colab.research.google.com/github/hemanthvnp/MvDA_Advancement/blob/main/notebooks/colab_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/hemanthvnp/MvDA_Advancement/blob/main/notebooks/colab_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MvDA on Colab

Multi-view Discriminant Analysis, reading **ColorFERET** straight from Google Drive (no manual download). If you upload a zip instead of using GitHub, skip cell 1 and `cd` into the folder.

## 1. Get the code

In [6]:
!git clone https://github.com/hemanthvnp/MvDA_Advancement.git mvda
%cd mvda
!pip -q install -r requirements.txt

Cloning into 'mvda'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 142 (delta 58), reused 130 (delta 49), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 278.63 KiB | 11.61 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/mvda


## 2. Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Locate ColorFERET and confirm it has images

If the image count is 0, your Drive copy is metadata-only and you need the real `.ppm`/`.ppm.bz2` files.

In [8]:
!find /content/drive/MyDrive -maxdepth 4 -iname '*colorferet*' -type d 2>/dev/null | head
FERET_ROOT = '/content/drive/MyDrive/colorferet'  # <-- set to the path printed above
!echo 'image files found:'; find "$FERET_ROOT" -type f \( -iname '*.ppm' -o -iname '*.ppm.bz2' -o -iname '*.png' -o -iname '*.jpg' \) 2>/dev/null | wc -l

/content/drive/MyDrive/colorferet
/content/drive/MyDrive/colorferet/colorferet
image files found:
50174


## 4. Sanity check on UCI Multiple Features (auto-downloads, no Drive needed)

In [4]:
!python experiments/run_mvda.py --dataset mfeat --mode concat --classifier ensemble

Loading dataset=mfeat scaler=robust ...
  views=6 dims=[76, 216, 64, 240, 47, 6] train=1000 test=1000 classes=10
Fitting MultiViewLDA(mode=concat, vc_lambda=0.0) ...
  shared-space dim = 9

=== mfeat | concat | ensemble ===
Accuracy: 98.700%

class   precision   recall      f1          support   
------------------------------------------------------
0       1.0000      0.9900      0.9950      100       
1       0.9709      1.0000      0.9852      100       
2       0.9900      0.9900      0.9900      100       
3       0.9796      0.9600      0.9697      100       
4       0.9901      1.0000      0.9950      100       
5       0.9898      0.9700      0.9798      100       
6       1.0000      0.9900      0.9950      100       
7       0.9804      1.0000      0.9901      100       
8       1.0000      0.9900      0.9950      100       
9       0.9703      0.9800      0.9751      100       
------------------------------------------------------
macro   0.9871      0.9870      0.9870    

## 5. ColorFERET, MvDA-paper protocol (subject-disjoint, 7 poses)

Kan et al. 2016: train on the first 231 identities (4 images/pose), then **gallery/probe recognition on the remaining, unseen identities** across 7 poses. First run reads + caches all images (slow); later runs reuse the cache.

In [9]:
!python experiments/run_feret.py --protocol disjoint --solver ratio \
    --feret-root "$FERET_ROOT" --feret-poses pl hl ql fa qr hr pr \
    --train-subjects 231 --images-per-pose 4 --gallery-pose fa \
    --feret-size 64 64 --pca 120 --save feret_paper_ratio.json

ColorFERET [disjoint] root=/content/drive/MyDrive/colorferet poses=['pl', 'hl', 'ql', 'fa', 'qr', 'hr', 'pr'] solver=ratio ...
  loading 501 subjects × 7 poses (501 tasks, 16 threads) ...
  subjects: 231 train / 270 test; poses=['pl', 'hl', 'ql', 'fa', 'qr', 'hr', 'pr']; gallery=fa
  shared-space dim = 230

Per-pose probe rank-1 accuracy:
pose    probes    acc       
----------------------------
pl      1197      10.53     
hl      1200      16.17     
ql      1200      23.50     
qr      1200      45.50     
hr      1200      22.75     
pr      1197      11.28     
----------------------------

=== ColorFERET | disjoint | ratio | rank-1 ===
Overall accuracy: 21.629%  (7194 probes, 270 subjects)
Macro F1: 0.1967


## 6. Does MvDA improve with the paper-based solvers?

Same paper protocol and split, swapping the discriminant solver:
- **`ratio`** — classical LDA (baseline).
- **`exponential`** — Exponential DA (robust to small-sample singularity, enlarges margins).
- **`harmonic`** — Harmonic-mean LDA (emphasizes confusable identity pairs).

Higher rank-1 accuracy for `exponential`/`harmonic` ⇒ the improvement holds on FERET.

In [10]:
for solver in ['ratio', 'exponential', 'harmonic']:
    print('\n================', solver, '================')
    !python experiments/run_feret.py --protocol disjoint --solver $solver \
        --feret-root "$FERET_ROOT" --feret-poses pl hl ql fa qr hr pr \
        --train-subjects 231 --images-per-pose 4 --gallery-pose fa \
        --feret-size 64 64 --pca 120


================ ratio ================
ColorFERET [disjoint] root=/content/drive/MyDrive/colorferet poses=['pl', 'hl', 'ql', 'fa', 'qr', 'hr', 'pr'] solver=ratio ...
  subjects: 231 train / 270 test; poses=['pl', 'hl', 'ql', 'fa', 'qr', 'hr', 'pr']; gallery=fa
  shared-space dim = 230

Per-pose probe rank-1 accuracy:
pose    probes    acc       
----------------------------
pl      1197      10.53     
hl      1200      16.17     
ql      1200      23.50     
qr      1200      45.50     
hr      1200      22.75     
pr      1197      11.28     
----------------------------

=== ColorFERET | disjoint | ratio | rank-1 ===
Overall accuracy: 21.629%  (7194 probes, 270 subjects)
Macro F1: 0.1967

================ exponential ================
ColorFERET [disjoint] root=/content/drive/MyDrive/colorferet poses=['pl', 'hl', 'ql', 'fa', 'qr', 'hr', 'pr'] solver=exponential ...
  subjects: 231 train / 270 test; poses=['pl', 'hl', 'ql', 'fa', 'qr', 'hr', 'pr']; gallery=fa
  shared-space dim = 23

## 7. (optional) Closed-set protocol

A simpler protocol: per-view split of all subjects, classify held-out single-pose images by nearest class mean.

In [ ]:
!python experiments/run_feret.py --protocol closed \
    --feret-root "$FERET_ROOT" --feret-poses fa fb hl hr --feret-size 64 64 --pca 120 --solver harmonic

ColorFERET [closed] root=/content/drive/MyDrive/colorferet poses=['fa', 'fb', 'hl', 'hr'] solver=harmonic ...
  loading 875 subjects × 4 poses (875 tasks, 16 threads) ...
